# AIシフト作成ツール

既存のシフト表Excel（.xlsm）の「個人シフト入力表」をAIが自動で埋めます。
上のセルから順に ▶ を押してください。

**はじめて使うとき**: 完成済みの月のExcelをアップロードし、モードを「学習」にして実行します。
スタッフごとの「使うシフト・働く曜日・出勤日数」を学習した `AI設定.xlsx` ができるので、ダウンロードして必要なら直してください。

**毎月の作成**: 新しい月のExcelと `AI設定.xlsx` をアップロードし、モードを「作成」にして実行します。

「個人シフト入力表」に入っている値は希望として扱います。

| 入力 | 意味 |
|---|---|
| シフトNO. | その日はそのシフトで確定 |
| 0 | 休み希望（確定） |
| 空欄 | AIが決める |

In [1]:
#@title ① 準備（最初に1回）
!pip -q install ortools openpyxl
import os, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/ngsw6126/AI_Schedule/main/shift_ai.py", "shift_ai.py")
import importlib, shift_ai
importlib.reload(shift_ai)
print("準備完了")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
準備完了


In [2]:
#@title ② ファイルをアップロード（Excel と、あれば AI設定.xlsx）
from google.colab import files
uploaded = files.upload()
excel = next((f for f in uploaded if f.lower().endswith((".xlsm", ".xlsx")) and "AI設定" not in f), None)
settings = next((f for f in uploaded if "AI設定" in f), "AI設定.xlsx" if os.path.exists("AI設定.xlsx") else None)
print("Excel:", excel)
print("設定:", settings or "なし（Excelの内容から学習します）")

Saving 【イオン佐野】202603スケジュール.xlsx to 【イオン佐野】202603スケジュール.xlsx
Excel: 【イオン佐野】202603スケジュール.xlsx
設定: なし（Excelの内容から学習します）


In [5]:
#@title ③ 実行
モード = "作成"  #@param ["作成", "学習"]
入力済みのシフトを消して全部作り直す = True  #@param {type:"boolean"}
計算時間_秒 = 60  #@param {type:"integer"}

if モード == "学習":
    shift_ai.main(["learn", excel, "-s", "AI設定.xlsx"])
    files.download("AI設定.xlsx")
else:
    args = ["generate", excel, "-o", "AIシフト完成_" + os.path.basename(excel),
            "-r", "AIシフト_レポート.xlsx", "-t", str(計算時間_秒)]
    if settings:
        args += ["-s", settings]
    if 入力済みのシフトを消して全部作り直す:
        args.append("--clear")
    out = shift_ai.main(args)
    files.download(out)
    files.download("AIシフト_レポート.xlsx")

2026年3月 / スタッフ11名 / シフト21種類 / 31日間 / 祝日[20]
結果: 最適解
人手不足の合計: 0 人時（不足なし）


スタッフ別:
  長澤　康雄: 24日 227.75時間（目標25日）
  関山　綺香: 22日 214.5時間（目標22日）
  飯田　正子: 1日 2.25時間（目標4日）
  福田　裕子: 5日 17.5時間（目標5日）
  奈良部　明夫: 10日 40時間（目標10日）
  岩﨑　好子: 7日 14時間（目標8日）
  秋山　紗那: 12日 63時間（目標12日）
  新倉　礼萌: 1日 6時間（目標10日）
  柴田　登志子: 18日 130.5時間（目標18日）
  安田　さくら: 5日 18時間（目標11日）
  川越　愛心: 7日 28時間（目標7日）

保存しました: AIシフト完成_【イオン佐野】202603スケジュール.xlsx
レポート: AIシフト_レポート.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>